<a href="https://colab.research.google.com/github/avikumart/LLM-GenAI-Transformers-Notebooks/blob/main/DeepLearningFiles/Transformer_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [3]:
# ─────────────────────────────────────────
# 1. Positional Encoding
# ─────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ─────────────────────────────────────────
# 2. Scaled Dot-Product Attention
# ─────────────────────────────────────────
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, S, S)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights


# ─────────────────────────────────────────
# 3. Multi-Head Attention
# ─────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, S, D = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, d_k)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(attn_out.size(0), -1, self.num_heads * self.d_k)
        return self.W_o(attn_out)


# ─────────────────────────────────────────
# 4. Feed-Forward Network
# ─────────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


# ─────────────────────────────────────────
# 5. Encoder Layer
# ─────────────────────────────────────────
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual
        x = self.norm1(x + self.drop(self.attn(x, x, x, mask)))
        # FFN + residual
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


# ─────────────────────────────────────────
# 6. Decoder Layer
# ─────────────────────────────────────────
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads)  # masked
        self.cross_attn = MultiHeadAttention(d_model, num_heads)  # encoder output
        self.ff         = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.drop(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.drop(self.ff(x)))
        return x


# ─────────────────────────────────────────
# 7. Full Transformer
# ─────────────────────────────────────────
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, max_len=5000, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)

        self.encoder = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.output_proj = nn.Linear(d_model, tgt_vocab)

    def encode(self, src, src_mask):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.src_embed.embedding_dim))
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.tgt_embed.embedding_dim))
        for layer in self.decoder:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encode(src, src_mask)
        dec_out = self.decode(tgt, enc_out, src_mask, tgt_mask)
        return self.output_proj(dec_out)  # (B, tgt_len, tgt_vocab)


# ─────────────────────────────────────────
# 8. Quick Test
# ─────────────────────────────────────────
if __name__ == "__main__":
    model = Transformer(src_vocab=1000, tgt_vocab=1000)
    src = torch.randint(0, 1000, (2, 10))   # (batch=2, src_len=10)
    tgt = torch.randint(0, 1000, (2, 8))    # (batch=2, tgt_len=8)
    out = model(src, tgt)
    print("Output shape:", out.shape)
    print(out)        # → (2, 8, 1000)

Output shape: torch.Size([2, 8, 1000])
tensor([[[ 0.0601, -0.0817, -0.5031,  ...,  1.1944, -0.3294, -0.5590],
         [-0.2947, -0.3092,  0.3309,  ..., -0.4151, -0.6969, -1.1849],
         [-0.4930, -0.6205, -0.0454,  ...,  0.0194,  0.4976, -0.5064],
         ...,
         [ 0.3857,  0.1528, -0.7555,  ...,  0.1828, -0.3633,  0.2541],
         [-0.2029, -0.8058, -0.8735,  ...,  0.0612,  0.5263, -0.2738],
         [-0.9422, -0.6691, -0.9472,  ..., -0.1293, -0.6030, -0.6507]],

        [[-0.6540, -0.2918, -0.4636,  ...,  0.5790,  0.2996,  0.3619],
         [ 0.7400, -1.2708, -0.5638,  ..., -0.2641,  0.4666,  0.0381],
         [-0.5682, -0.5173,  0.5712,  ...,  0.6055, -1.3435, -0.5991],
         ...,
         [-0.7299, -1.8222,  0.4287,  ...,  0.0945,  0.3941, -0.5765],
         [ 0.3223, -0.1589,  0.2967,  ...,  0.5476,  0.5952, -0.1366],
         [ 0.3351, -0.4229, -0.5956,  ...,  1.0420, -0.3129, -0.4242]]],
       grad_fn=<ViewBackward0>)
